## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [2]:
MODEL = "gemma3:1b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
MODEL = "gemma3:1b"
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='2a1fef85-563c-47dd-92a8-be329fecb9b6', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery is a hugely popular and enigmatic figure in the world of online gaming, particularly within the Fortnite community. Here\'s a breakdown of who she is and why she\'s become so well-known:\n\n**Who is Avery?**\n\n* **A Former Fortnite Streamer:** Avery was a prominent Fortnite streamer for several years, known for her engaging personality, humor, and unique gameplay style.\n* **The "Ghost" Persona:** She adopted a persona called "Ghost" – a mysterious, almost ethereal figure – during her streams. This persona was incredibly popular and drew a massive following.\n* **A Mysterious and Artistic Style:** Avery\'s "Ghost" persona was characterized by a distinctive, almost painterly aesthetic. She used a combination of pixel art, stylized animations, and a calming, ethereal color palette.  She often created scenes that felt like a digital painting.\n* **Unique Content & Storytelling:** Avery wasn\'t just playing Fortnite; she crafted elaborate narratives and stories wi

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

"Okay, let's talk about Avery Lancaster!\n\nAvery Lancaster is the **Co-Founder and Chief Executive Officer (CEO)** of Insurellm. She’s a key figure in the company’s leadership and has been instrumental in its growth and success.\n\nHere’s a breakdown of what we know about her:\n\n*   **Role:** CEO of Insurellm\n*   **Experience:** She’s been with the company since 2015 and has been leading the charge since its inception.\n*   **Background:** She previously worked as a Senior Product Manager at Innovate Insurance Solutions, where she was responsible for developing innovative insurance products.\n*   **Recognition:** She was recognized as Insurellm’s Innovator of the year in 2023.\n\nEssentially, Avery is the driving force behind Insurellm’s vision and strategy.\n\nDo you have any specific questions about Avery that you’d like me to answer?"

## What could possibly come next? 😂

In [19]:
gr.ChatInterface(answer_question).launch()

c:\JM\LLM_course\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!